# Привязать растры TanDEM-X

Запустите первую ячейку

In [11]:
import arcpy
import xml.etree.ElementTree as ET
import re

from pathlib import Path
from typing import Any, Optional

H_WKID_WGS84 = 4326
V_WKID_WGS84 = 115700
V_WKID_EGM = 5773

def collect_tile_info(base_path: Path) -> list[dict[str, Any]]:
    """
    Обходит папки с тайлами TanDEM-X и собирает данные для дальнейшей их геопривязки.

    :param base_path: Путь к тайлам TanDEM-X.
    
    """
    tile_re = re.compile(r"(TDM1_EDEM_10_[N,S]\d+[E,W]\d+)_V\d+_C")
    tiles_descr: list[dict[str, Any]] = []
    
    if not base_path.exists():
        return []

    for folder in [d for d in base_path.iterdir() if d.is_dir()]:
        match = tile_re.match(folder.name)
        if match:
            tile_id = match.group(1)
            xml_path = folder / f"{tile_id}.xml"
            
            if xml_path.exists():
                metadata = get_spatial_metadata(xml_path)
                edem_folder = folder / "EDEM"
                tiles_descr.append({
                    'tile_id': tile_id,
                    'folder_name': folder.name,
                    'metadata': metadata,
                    'rasters': [
                        {
                            'path': edem_folder / f"{tile_id}_EDEM_EGM.tif", 
                            'v_WKID': V_WKID_EGM,
                            'type': 'EGM'
                        },
                        {
                            'path': edem_folder / f"{tile_id}_EDEM_W84.tif", 
                            'v_WKID': V_WKID_WGS84,
                            'type': 'WGS84'
                        }
                    ]
                })
    return tiles_descr


def get_spatial_metadata(xml_path: Path) -> Optional[dict[str, Any]]:
    '''
    Извлечь пространственную привязку из XML файла описания тайла.
    
    :param xml_path: Путь к xml файлу описания тайла.

    :returns: Метаданные для привязки растров.
    '''
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        get_val = lambda tag: float(root.find(f".//{tag}").text)
        return {
            'ul': (get_val('upperLeftLongitude'), get_val('upperLeftLatitude')),
            'bbox': (get_val('westBoundingCoordinate'), get_val('southBoundingCoordinate'), 
                     get_val('eastBoundingCoordinate'), get_val('northBoundingCoordinate'))
        }
    except:
        return None

def apply_georeference(raster_path: Path, metadata: Optional[dict[str, Any]], h_WKID: Any= H_WKID_WGS84, v_WKID: Any = V_WKID_EGM) -> bool | Exception:
    '''
    Геопривязка растра: создание .tfw и назначение системы координат.

    :param raster_path: Путь к растру.
    :param metadata: Метаданные для привязки растров из функции `get_spatial_metadata`.
    :param h_WKID: идентификатор горизонтальной СК, например, 4326 — WGS84.
    :param v_WKID: идентификатор вертикальной СК, например, 5773 — EGM, 115700 — эллипсоидальная WGS 1984.

    :returns: Удалось ли присвоить систему координат.
    '''
    if not raster_path.exists() or metadata is None: 
        return False
    
    try:
        
        # Расчитать параметры привязки
        desc = arcpy.Describe(str(raster_path)) # arcpy.Describe для одноканального растра возвращает такаже свойства единственного канала растра
        ul_lon, ul_lat = metadata['ul']
        west, south, east, north = metadata['bbox']        
        # Расчет размера пикселя в системе координат (пространственное разрешение)
        px_x, px_y = (east - west) / desc.width, (north - south) / desc.height    
        # В XML-файле координаты upperLeftLongitude/Latitude указывают на внешний угол (край) самого крайнего пикселя.
        # Формат World File (.tfw) требует координаты центра верхнего левого пикселя.
        c_x, c_y = ul_lon + (px_x / 2), ul_lat - (px_y / 2)

        # Создание World File
        tfw_path = raster_path.with_suffix('.tfw')
        tfw_path.write_text('\n'.join([
            f"{px_x:.12f}", # Масштаб по оси X
            "0.0",          # Поворот (Y-axis rotation)
            "0.0",          # Поворот (X-axis rotation)
            f"{-px_y:.12f}",# Масштаб по оси Y. Важно: он всегда отрицательный, так как в компьютерных изображениях координаты Y растут сверху вниз, а в географии широта растет снизу вверх 
            f"{c_x:.12f}",  # Координата X центра верхнего левого пикселя
            f"{c_y:.12f}"   # Координата Y центра верхнего левого пикселя
        ]))

        # Присвоить систему координат
        sr = arcpy.SpatialReference(h_WKID, v_WKID)
        arcpy.management.DefineProjection(str(raster_path), sr)
        
        return True
    except Exception as e:
        return e

def process_tiles(tiles_descr: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """
    Выполняет привязку тайлов.

    :param tiles_descr: Информация о тайле.

    :returns: Информация о результатах обработки.
    
    """
    results: list[dict[str, Any]] = []
    arcpy.env.overwriteOutput = True
    
    for tile in tiles_descr:
        tile_results = {            
            'tile_id': tile['tile_id'], 
            'folder': tile['folder_name'], 
            'EGM': False, 
            'WGS84': False,
            'descr': tile
        }
        
        for raster in tile['rasters']:
            run_res = apply_georeference(
                raster_path=raster['path'], 
                metadata=tile['metadata'], 
                v_WKID=raster['v_WKID']
            )
            if isinstance(run_res, bool):
                tile_results[raster['type']] = run_res
            else:
                tile_results['Error'] = str(run_res)
                tile_results['ErrorType'] = type(run_res)
                
            
        results.append(tile_results)
    return results

def print_report(results: list[dict[str, Any]]) -> None:
    """
    Выводит отчет об привязке тайлов TanDEM-X.
    
    """
    
    if not results:
        print("Тайлы TanDEM-X для обработки не найдены.")
        return

    header: str = f"{'Тайл TanDEM-X':<40} | {'DEM EGM':<9} | {'DEM WGS84':<9}"
    print(header)
    print("-" * len(header))
    
    for r in results:
        egm_stat = "OK" if r['EGM'] else "--"
        w84_stat = "OK" if r['WGS84'] else "--"
        print(f"{r['folder']:<40} | {egm_stat:<9} | {w84_stat:<9}")
        
    print("-" * len(header))
        
    success_egm = sum(1 for r in results if r['EGM'])
    success_w84 = sum(1 for r in results if r['WGS84'])
    
    print(f"ИТОГО: Успешно привязано тайлов TanDEM-X для вертикальной СК EGM: {success_egm}, для вертикальной СК WGS84: {success_w84}")


## Пользовательская конфигурация

Внесите в следующей ячейке в переменную **input_folder** путь к папке с тайлами Введите путь к папке с данными. После изменения запустите следующую ячейку.

In [12]:
# Укажите путь к папке Maps\TanDEM-X
input_folder = Path(arcpy.mp.ArcGISProject("CURRENT").filePath).parent / "TanDEM-X"

if input_folder.exists():
    print(f"Путь указан корректно. Можно запускать обработку.")
else:
    print("ВНИМАНИЕ: Указанный путь не найден.")
    

Путь указан корректно. Можно запускать обработку.


## Запуск
Запустите следующую ячейку для геопривязки тайлов TanDEM-X (генерация .tfw и установка систем координат).

In [13]:
process_result = process_tiles(collect_tile_info(input_folder))
print_report(process_result)

Тайл TanDEM-X                            | DEM EGM   | DEM WGS84
----------------------------------------------------------------
TDM1_EDEM_10_N04E005_V01_C               | OK        | OK       
TDM1_EDEM_10_N04E006_V01_C               | OK        | OK       
TDM1_EDEM_10_N04W002_V01_C               | OK        | OK       
TDM1_EDEM_10_N05E000_V01_C               | OK        | OK       
TDM1_EDEM_10_N05E001_V01_C               | OK        | OK       
TDM1_EDEM_10_N05E005_V01_C               | OK        | OK       
TDM1_EDEM_10_N05E006_V01_C               | OK        | OK       
TDM1_EDEM_10_N05W001_V01_C               | OK        | OK       
TDM1_EDEM_10_N05W002_V01_C               | OK        | OK       
TDM1_EDEM_10_N06E000_V01_C               | OK        | OK       
TDM1_EDEM_10_N06E001_V01_C               | OK        | OK       
TDM1_EDEM_10_N06E002_V01_C               | OK        | OK       
TDM1_EDEM_10_N06E003_V01_C               | OK        | OK       
TDM1_EDEM_10_N06E004_V01_

## Набор данных мозаики ArcGIS
### Конфигурация
Если Вы хотите создать мозаику из растров, то введите путь к GDB **target_gdb** и запустите следующую ячейку для подготовки ГБД.

In [14]:
# Путь к базе данных (GDB)
target_gdb = Path(arcpy.mp.ArcGISProject("CURRENT").filePath).parent / "TanDEMX.gdb"

if not arcpy.Exists(target_gdb):
    folder, name = target_gdb.parent, target_gdb.name
    arcpy.management.CreateFileGDB(str(folder), str(name))
    print(f"Создана новая база данных: {target_gdb}")
else:
    print(f"Используется существующая БД: {target_gdb}")

Создана новая база данных: I:\docs\maxim\prj\gis\learn\Наводнения Бенин (Смаил)\Maps\TanDEMX.gdb


### Запуск создания набора данных мозаики
Укажите название мозаики **mosaic_names** и запустите следующую ячейку для создания Mosaic Dataset и автоматически добавления в него всех растров.

In [15]:
# Имя мозаики (лучше указывать тип данных, например _EGM)
mosaic_names = {
    'EGM': "TDX_EGM_Mosaic",
    'WGS84': "TDX_W84_Mosaic"
}

srs = {
    'EGM': arcpy.SpatialReference(H_WKID_WGS84, V_WKID_EGM), 
    'WGS84': arcpy.SpatialReference(H_WKID_WGS84, V_WKID_WGS84)
}

rasters = {
    'EGM': [
        str(raster['path'])
        for result in process_result if result.get('EGM')
        for raster in result['descr']['rasters'] if raster['type'] == 'EGM'
    ],
    'WGS84': [
        str(raster['path'])
        for result in process_result if result.get('WGS84')
        for raster in result['descr']['rasters'] if raster['type'] == 'WGS84'
    ]
}

for v_crs in mosaic_names:
    if not rasters[v_crs]:
        continue

    mosaic_path = target_gdb / mosaic_names[v_crs]
    
    print(f"1/2: Создание новой Mosaic Dataset ({v_crs})…")
    if arcpy.Exists(str(mosaic_path)):
        arcpy.management.Delete(str(mosaic_path))
    arcpy.management.CreateMosaicDataset(str(target_gdb), mosaic_names[v_crs], srs[v_crs])
    
    print(f"2/2: Добавление растров ({v_crs})…")
    arcpy.management.AddRastersToMosaicDataset(
        str(mosaic_path),
        "Raster Dataset", 
        rasters[v_crs],
        duplicate_items_action="EXCLUDE_DUPLICATES",
        update_boundary="UPDATE_BOUNDARY",
        update_overviews="UPDATE_OVERVIEWS", # обзорные изображения набора данных мозаики
        build_pyramids="BUILD_PYRAMIDS", # Строит внутренние пирамиды для каждого файла
        calculate_statistics="CALCULATE_STATISTICS",
        estimate_statistics="ESTIMATE_STATISTICS"
    )
        
    print(f"Мозаика '{mosaic_names[v_crs]}' готова к работе в ArcGIS Pro.")

print(f"\nГотово!")

1/2: Создание новой Mosaic Dataset (EGM)…
2/2: Добавление растров (EGM)…
Мозаика 'TDX_EGM_Mosaic' готова к работе в ArcGIS Pro.
1/2: Создание новой Mosaic Dataset (WGS84)…
2/2: Добавление растров (WGS84)…
Мозаика 'TDX_W84_Mosaic' готова к работе в ArcGIS Pro.

Готово!
